In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.ndimage import gaussian_filter1d

In [19]:
# Cargar datos
data = []
with open("velocities_a21.xvg", "r") as f:
    for line in f:
        if line.startswith("#") or line.startswith("@"):
            continue
        cols = line.split()
        if len(cols) >= 4:
            data.append([float(c) for c in cols])

data = np.array(data)
vx = data[:, 1]
vy = data[:, 2]
vz = data[:, 3]

# Cargar temperatura
temp_data = []
with open("temp.xvg", "r") as f:
    for line in f:
        if line.startswith("#") or line.startswith("@"):
            continue
        cols = line.split()
        if len(cols) >= 2:
            temp_data.append(float(cols[1]))

T = np.array(temp_data)

# Función para generar las gráficas
def plot_hist(vel, label, color, filename, xlabel, T_ref=None):
    fig, ax = plt.subplots(figsize=(8, 6))

    # Histograma
    counts, bins, _ = ax.hist(vel, bins=100,
                               color=color, alpha=0.5,
                               edgecolor=color, label="Datos")

    # KDE
    kde = gaussian_kde(vel)
    x = np.linspace(vel.min(), vel.max(), 300)
    scaling = len(vel) * (bins[1] - bins[0])
    ax.plot(x, kde(x) * scaling,
            color="darkgreen", linewidth=2, label="KDE")

    # Gaussiana ajustada en negro
    mu, sigma = norm.fit(vel)
    ax.plot(x, norm.pdf(x, mu, sigma) * scaling,
            color="black", linewidth=2, linestyle="--",
            label=f"Gaussiana (μ={mu:.3f}, σ={sigma:.3f})")

    if T_ref is not None:
        ax.axvline(T_ref, color="blue", linestyle="--",
                   linewidth=1.5, label=f"T objetivo ({T_ref} K)")

    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel("Frecuencia", fontsize=13)
    ax.set_title(f"Histograma {label} — Cα PRO-3", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Guardado: {filename}")

# Generación de las gráficas
plot_hist(vx, "Vx", "green",   "hist_vx.png",           "Vx (nm/ps)")
plot_hist(vy, "Vy", "green",   "hist_vy.png",           "Vy (nm/ps)")
plot_hist(vz, "Vz", "green",   "hist_vz.png",           "Vz (nm/ps)")
plot_hist(T,  "Temperatura", "crimson", "hist_temperatura.png", "Temperatura (K)", T_ref=298)

Guardado: hist_vx.png
Guardado: hist_vy.png
Guardado: hist_vz.png
Guardado: hist_temperatura.png
